In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Point

In [2]:
# 1. Load ward shapefile
wards = gpd.read_file(r"E:\SQL learning 2\airbnb_project\Bristol\download\shapefiles")
print(wards.columns)   # see what attributes are available (e.g. ward name column)

# 2. Load your listings table (with latitude & longitude columns)
df = pd.read_csv("E:\SQL learning 2\\airbnb_project\Bristol\download\csv_file\Bristol_listings.csv", encoding="latin1")  # replace with your table file
print(df.head())

# 3. Convert listings into GeoDataFrame of points
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),  # adjust column names if different
    crs="EPSG:4326"   # WGS84 (lat/lon)
)

# 4. Make sure shapefile is in the same CRS (coordinate system)
wards = wards.to_crs(epsg=4326)

# 5. Spatial join → assign each listing a ward
joined = gpd.sjoin(gdf_points, wards, how="left", predicate="within")
print(joined.columns.tolist())

# 6. Inspect result
print(joined[["id", "price"]].head())  # replace with actual ward name column


Index(['OBJECTID', 'NAME', 'COUNCILLOR', 'WARD_ID', 'SHAPESTAre', 'SHAPESTLen',
       'geometry'],
      dtype='object')
             id                                       listing_url  \
0  2.179747e+07             https://www.airbnb.com/rooms/21797474   
1  4.543449e+07             https://www.airbnb.com/rooms/45434492   
2  7.971370e+17   https://www.airbnb.com/rooms/797137345244026756   
3  1.313700e+18  https://www.airbnb.com/rooms/1313695117463755982   
4  5.002188e+07             https://www.airbnb.com/rooms/50021882   

      scrape_id last_scraped           source  \
0  2.025030e+13   19/03/2025      city scrape   
1  2.025030e+13   19/03/2025      city scrape   
2  2.025030e+13   19/03/2025      city scrape   
3  2.025030e+13   19/03/2025      city scrape   
4  2.025030e+13   19/03/2025  previous scrape   

                                         name  \
0  7 Southover Close Westbury On Trym Bristol   
1                Double room in peaceful home   
2              Bright

In [3]:
# After the spatial join
print(joined.columns.tolist())   # to confirm exact names

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availabil

In [17]:
# Example: compare boroughs
joined["match"] = joined["neighbourhood_cleansed"] == joined["NAME"]

# See mismatches
matches = joined[joined["match"] == True]
mismatches = joined[joined["match"] == False]
print(matches[["id", "neighbourhood_cleansed", "NAME"]])
print('There are {} mismatches'.format(len(mismatches)))


                id       neighbourhood_cleansed                         NAME
0     2.179747e+07  Westbury-on-Trym & Henleaze  Westbury-on-Trym & Henleaze
1     4.543449e+07             Brislington West             Brislington West
2     7.971370e+17                      Clifton                      Clifton
3     1.313700e+18                      Clifton                      Clifton
4     5.002188e+07                       Easton                       Easton
...            ...                          ...                          ...
2767  1.187050e+18             Brislington East             Brislington East
2768  8.436520e+17                       Ashley                       Ashley
2769  1.355060e+18                      Central                      Central
2770  6.982980e+17                Windmill Hill                Windmill Hill
2771  1.295540e+18                Lawrence Hill                Lawrence Hill

[2772 rows x 3 columns]
There are 0 mismatches
